# ChartQAPro Evaluation — Analysis Walkthrough

This notebook walks through the full evaluation artifact stack produced by the
agentic ChartQAPro framework. By the end you will be able to:

- Load and inspect **MEPs** (Model Evaluation Packets) directly
- Plot **accuracy by question type** from `metrics.jsonl`
- Visualise the **verifier revision rate** and its effect on accuracy
- Chart the **failure taxonomy** breakdown from `taxonomy.jsonl`
- Browse individual samples — question, plan, vision answer, verifier verdict, chart image

**Prerequisites:** Run these commands first:
```bash
python -m agentic_chartqapro_eval.runner.run_generate_meps --n 25 --config gemini_gemini
python -m agentic_chartqapro_eval.eval.eval_outputs  --mep_dir meps/gemini_gemini/chartqapro/test --out output/metrics.jsonl
python -m agentic_chartqapro_eval.eval.error_taxonomy --mep_dir meps/gemini_gemini/chartqapro/test --metrics_file output/metrics.jsonl --out output/taxonomy.jsonl
```

## 1. Setup

In [ ]:
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

# Point these at your actual output files
MEP_DIR      = Path('./meps/gemini_gemini/chartqapro/test')
METRICS_FILE = Path('./output/metrics.jsonl')
TAXONOMY_FILE = Path('./output/taxonomy.jsonl')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('Setup complete')

## 2. Load MEPs

Each MEP is a self-contained JSON file capturing the full agent trace for one sample.
We read them all into a list of dicts for analysis.

In [ ]:
meps = []
for path in sorted(MEP_DIR.glob('*.json')):
    meps.append(json.loads(path.read_text()))

print(f'Loaded {len(meps)} MEPs from {MEP_DIR}')

# Peek at one MEP's top-level keys
if meps:
    print('\nTop-level MEP keys:', list(meps[0].keys()))
    print('Plan steps for first MEP:')
    for i, step in enumerate(meps[0].get('plan', {}).get('parsed', {}).get('steps', []), 1):
        print(f'  {i}. {step}')

## 3. Load metrics.jsonl

`metrics.jsonl` is produced by `eval_outputs.py` and contains one row per sample
with accuracy, latency, verifier verdict, and (optionally) judge scores.

In [ ]:
metrics = [json.loads(l) for l in METRICS_FILE.read_text().splitlines() if l.strip()]
df = pd.DataFrame(metrics)

print(f'Columns: {list(df.columns)}')
print(f'\nSamples: {len(df)}')
print(f'Overall accuracy: {df["answer_accuracy"].mean():.1%}')
df[['answer_accuracy', 'latency_sec']].describe().round(3)

## 4. Accuracy by Question Type

ChartQAPro has five question types: `standard`, `mcq`, `conversational`,
`hypothetical`, `unanswerable`. Performance varies significantly across types —
this plot shows where the model struggles.

In [ ]:
by_type = df.groupby('question_type')['answer_accuracy'].agg(['mean', 'count']).reset_index()
by_type = by_type.sort_values('mean', ascending=True)

colors = ['#d63031' if v < 0.4 else '#fdcb6e' if v < 0.75 else '#00b894'
          for v in by_type['mean']]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(by_type['question_type'], by_type['mean'], color=colors, edgecolor='white')

for bar, (_, row) in zip(bars, by_type.iterrows()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{row['mean']:.1%}  (n={int(row['count'])})",
            va='center', fontsize=10)

ax.set_xlim(0, 1.15)
ax.set_xlabel('Answer Accuracy')
ax.set_title('Accuracy by Question Type', fontweight='bold')
ax.axvline(df['answer_accuracy'].mean(), color='#6c5ce7', linestyle='--', linewidth=1.2, label='Overall mean')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Verifier (Pass 2.5) Analysis

The VerifierAgent independently re-examines each chart image and either **confirms**
or **revises** the VisionAgent's draft answer. This cell shows:
- How often the verifier intervened
- Whether revisions improved accuracy

In [ ]:
if 'verifier_verdict' not in df.columns or df['verifier_verdict'].eq('skipped').all():
    print('Verifier was not used (--no_verifier flag was set). Re-run without it to see this analysis.')
else:
    verdict_counts = df['verifier_verdict'].value_counts()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    
    # Pie chart: verdict distribution
    verdict_colors = {'confirmed': '#00b894', 'revised': '#fdcb6e', 'skipped': '#b2bec3'}
    vc = verdict_counts.reindex(['confirmed', 'revised', 'skipped']).dropna()
    ax1.pie(vc.values, labels=vc.index,
            colors=[verdict_colors[k] for k in vc.index],
            autopct='%1.1f%%', startangle=90)
    ax1.set_title('Verifier Verdict Distribution', fontweight='bold')
    
    # Bar chart: accuracy by verdict
    acc_by_verdict = df.groupby('verifier_verdict')['answer_accuracy'].mean()
    bars = ax2.bar(acc_by_verdict.index,
                   acc_by_verdict.values,
                   color=[verdict_colors.get(k, '#6c5ce7') for k in acc_by_verdict.index],
                   edgecolor='white', width=0.5)
    for bar in bars:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.1%}', ha='center', fontsize=10)
    ax2.set_ylim(0, 1.1)
    ax2.set_ylabel('Avg Answer Accuracy')
    ax2.set_title('Accuracy by Verifier Verdict', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Also show before/after for revised samples
    revised_rows = df[df['verifier_verdict'] == 'revised']
    if len(revised_rows) > 0 and 'vision_answer' in df.columns:
        from agentic_chartqapro_eval.eval.eval_outputs import score_answer_accuracy
        before = revised_rows.apply(
            lambda r: score_answer_accuracy(r['expected'], r['vision_answer'], r['question_type']), axis=1
        ).mean()
        after = revised_rows['answer_accuracy'].mean()
        print(f'\nFor the {len(revised_rows)} revised samples:')
        print(f'  Accuracy BEFORE verifier: {before:.1%}')
        print(f'  Accuracy AFTER  verifier: {after:.1%}')
        print(f'  Delta: {after - before:+.1%}')

## 6. Failure Taxonomy (Pass 4)

The `error_taxonomy.py` pass asks a VLM to diagnose *why* the agent failed — using
the chart image, the wrong answer, the correct answer, and the agent's explanation.
This gives a grounded diagnosis rather than a guess.

In [ ]:
if not TAXONOMY_FILE.exists():
    print(f'{TAXONOMY_FILE} not found. Run error_taxonomy.py first.')
else:
    tax = [json.loads(l) for l in TAXONOMY_FILE.read_text().splitlines() if l.strip()]
    tax_df = pd.DataFrame(tax)
    
    counts = tax_df['failure_type'].value_counts()
    
    palette = {
        'correct':                '#00b894',
        'extraction_error':       '#e84393',
        'axis_misread':           '#e17055',
        'arithmetic_mistake':     '#fdcb6e',
        'legend_confusion':       '#fd79a8',
        'hallucinated_element':   '#d63031',
        'unanswerable_failure':   '#0984e3',
        'question_misunderstanding': '#6c5ce7',
        'other':                  '#b2bec3',
    }
    bar_colors = [palette.get(k, '#6c5ce7') for k in counts.index]
    
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(counts.index[::-1], counts.values[::-1],
                   color=bar_colors[::-1], edgecolor='white')
    for bar in bars:
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                str(int(bar.get_width())), va='center', fontsize=10)
    ax.set_xlabel('Number of samples')
    ax.set_title('Failure Taxonomy Breakdown (Pass 4)', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    wrong = tax_df[tax_df['failure_type'] != 'correct']
    print(f'\nTotal wrong: {len(wrong)} / {len(tax_df)}')
    print(f'Most common failure: {counts[counts.index != "correct"].idxmax()}')

## 7. Judge Scores (if available)

When `eval_outputs.py` is run without `--no_judge`, five rubric scores are computed
per sample by an LLM judge. This cell plots their distributions.

In [ ]:
judge_cols = [c for c in df.columns if c.startswith('judge_') and c != 'judge_parse_error']

if not judge_cols:
    print('No judge scores found. Re-run eval_outputs.py without --no_judge.')
else:
    fig, axes = plt.subplots(1, len(judge_cols), figsize=(3.5 * len(judge_cols), 3.5), sharey=False)
    if len(judge_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, judge_cols):
        vals = pd.to_numeric(df[col], errors='coerce').dropna()

        ax.hist(vals, bins=10, range=(0, 1), color='#6c5ce7', edgecolor='white', alpha=0.85)

        if len(vals) > 0:
            mean_val = vals.mean()
            ax.axvline(mean_val, color='#d63031', linestyle='--', linewidth=1.5)
            ax.text(0.05, ax.get_ylim()[1] * 0.9, f'μ={mean_val:.2f}', fontsize=9, color='#d63031')

        ax.set_title(col.replace('judge_', '').replace('_', '\n'), fontsize=9, fontweight='bold')
        ax.set_xlabel('Score (0–1)')

    plt.suptitle('LLM Judge Rubric Distributions', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 8. Browse a Single MEP

Change `SAMPLE_ID` below to any sample ID in your MEP directory.
This cell prints the full agent trace and displays the chart image.

In [ ]:
from IPython.display import Image, display

# ← change to any sample_id in your MEP directory
SAMPLE_ID = meps[0]['sample']['sample_id'] if meps else None

mep = next((m for m in meps if m['sample']['sample_id'] == SAMPLE_ID), None)
if mep is None:
    print(f'Sample {SAMPLE_ID!r} not found in {MEP_DIR}')
else:
    sample   = mep['sample']
    plan     = mep.get('plan', {}).get('parsed', {})
    vision   = mep.get('vision', {}).get('parsed', {})
    verifier = mep.get('verifier') or {}
    v_parsed = verifier.get('parsed', {})

    print('=' * 60)
    print(f"Sample : {sample['sample_id']}")
    print(f"Type   : {sample['question_type']}")
    print(f"Q      : {sample['question']}")
    print(f"Expected: {sample['expected_output']}")
    print()
    print('--- Planner steps ---')
    for i, s in enumerate(plan.get('steps', []), 1):
        print(f'  {i}. {s}')
    print()
    print('--- VisionAgent ---')
    print(f"  Draft answer : {vision.get('answer', '—')}")
    print(f"  Explanation  : {vision.get('explanation', '—')[:200]}")
    print()
    if v_parsed:
        print('--- VerifierAgent ---')
        print(f"  Verdict  : {v_parsed.get('verdict', '—')}")
        print(f"  Answer   : {v_parsed.get('answer', '—')}")
        print(f"  Reasoning: {v_parsed.get('reasoning', '—')}")
    print('=' * 60)

    img_path = sample.get('image_ref', {}).get('path', '')
    if img_path and Path(img_path).exists():
        display(Image(filename=img_path, width=600))
    else:
        print(f'Image not found at: {img_path}')

## 9. Failure Cases Deep Dive

Find all samples where the verifier revised the answer but it was still wrong —
these are the hardest cases where neither agent got it right.

In [ ]:
if 'verifier_verdict' in df.columns:
    revised_wrong = df[
        (df['verifier_verdict'] == 'revised') &
        (df['answer_accuracy'] < 1.0)
    ][['sample_id', 'question_type', 'expected', 'vision_answer', 'predicted', 'answer_accuracy']]
    
    print(f'Samples where verifier revised but was still wrong: {len(revised_wrong)}')
    if len(revised_wrong) > 0:
        display(revised_wrong.head(10))
else:
    print('verifier_verdict column not present — run with verifier enabled')